# 04 Verify RBAC against the role catalog

Job task 4 of 5. Nothing in the access-control layer was ever checked before:
this notebook compares the **live** catalog state with `rbac/role_catalog.json`.

Checks:
1. every catalog group exists (warn when it is only an account group)
2. every group has `USE CATALOG` and exactly the `USE SCHEMA` set the catalog lists
3. schema-level `SELECT` / `MODIFY` exist only where the catalog allows them, and where it requires them
4. domain-scoped roles hold `SELECT` on exactly the tables tagged with their domains
5. every column tagged direct identifier / quasi identifier / HL7 V_Restricted has a mask, in
   every schema listed in `masking.masked_schemas` (gold and silver), or is in
   `masking.deferred_columns` / `masking.exempt_columns` for that schema
6. every gold table with an RLS anchor column has a row filter, and the uncovered tables are listed

Grants come from the `information_schema` privilege views (`catalog_privileges`,
`schema_privileges`, `table_privileges`): one query per level, every principal in one
pass. `SHOW GRANTS` has no `TO <principal>` form - the documented syntax is
`SHOW GRANTS [principal] ON <securable>`, which would need one query per table.

The views report privileges granted on the object. If a group is a member of
another group, inherited privileges can appear as "extra" here; the catalog deliberately
keeps the groups flat, so this is a drift signal rather than a false alarm.

In [0]:
import json
import os

dbutils.widgets.text(
    "catalog_path",
    "/Workspace/Shared/RMIT_CDR_Capstone/04_Governance/rbac/role_catalog.json",
)
dbutils.widgets.dropdown("fail_on_gap", "true", ["true", "false"])

CATALOG_PATH = dbutils.widgets.get("catalog_path")
FAIL_ON_GAP = dbutils.widgets.get("fail_on_gap") == "true"


def load_catalog(path):
    candidates = [
        path,
        "04_Governance/rbac/role_catalog.json",
        os.path.join(os.getcwd(), "04_Governance/rbac/role_catalog.json"),
    ]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            with open(candidate, "r", encoding="utf-8") as handle:
                return json.load(handle), candidate
    raise FileNotFoundError("role_catalog.json not found: " + ", ".join(candidates))


CATALOG_DEF, CATALOG_SOURCE = load_catalog(CATALOG_PATH)
CATALOG = CATALOG_DEF["catalog"]
GOLD = CATALOG_DEF["gold_schema"]
ROLES = CATALOG_DEF["roles"]
MASKING = CATALOG_DEF["masking"]
MASKED_SCHEMAS = [str(schema).lower() for schema in MASKING.get("masked_schemas", [GOLD])]
SCHEMA_LIST = ", ".join(f"'{schema}'" for schema in MASKED_SCHEMAS)
print(f"{len(ROLES)} roles from {CATALOG_SOURCE}; masked schemas {MASKED_SCHEMAS}")

7 roles from /Workspace/Shared/RMIT_CDR_Capstone/04_Governance/rbac/role_catalog.json; masked schemas ['gold', 'silver']


In [0]:
gaps = []
notes = []


def check(label, ok, detail=""):
    print(("PASS  " if ok else "GAP   ") + label + (f": {detail}" if detail else ""))
    if not ok:
        gaps.append(label)


def note(label, detail=""):
    print("NOTE  " + label + (f": {detail}" if detail else ""))
    notes.append(label)


def row_dict(row):
    return {str(k): str(v) for k, v in row.asDict().items()}


def pick(record, *names, default=""):
    for name in names:
        if name in record and record[name] not in ("None", ""):
            return record[name]
    return default


## 1. Groups

In [0]:
try:
    workspace_groups = {str(row.name) for row in spark.sql("SHOW GROUPS").collect()}
except Exception as exc:
    workspace_groups = set()
    note("could not list workspace groups", str(exc).splitlines()[0][:160])

for role_name in ROLES:
    if role_name in workspace_groups:
        check(f"group {role_name} exists (workspace-local)", True)
    else:
        note(f"group {role_name} not workspace-local (expected if account-managed)")

for group in sorted(workspace_groups):
    if group.startswith(("cdr_", "role_")) and group not in ROLES:
        note(f"group {group} is not in the role catalog")

PASS  group cdr_admins exists (workspace-local)
PASS  group cdr_standard_researcher exists (workspace-local)
PASS  group cdr_researcher_psychiatry exists (workspace-local)
PASS  group cdr_pharmacy_analyst exists (workspace-local)
PASS  group cdr_external_researcher exists (workspace-local)
PASS  group cdr_data_scientist exists (workspace-local)
PASS  group cdr_data_engineers exists (workspace-local)


## 2-4. Grants

In [0]:
domain_rows = spark.sql(f"""
    SELECT g.table_name, g.tag_value
    FROM {CATALOG}.information_schema.table_tags g
    WHERE g.schema_name = '{GOLD}' AND g.tag_name = 'domain'
""").collect()

tables_by_domain = {}
for row in domain_rows:
    tables_by_domain.setdefault(str(row["tag_value"]), set()).add(str(row["table_name"]))


def normalised(record):
    """Row dict with column names lower-cased and underscores removed.

    Databricks returns camelCase column names for some result sets (`actionType`) and
    snake_case for others (`privilege_type`); one shape for both so a lookup cannot miss.
    """
    return {str(k).lower().replace("_", ""): str(v) for k, v in record.items()}


def field(record, *names, default=""):
    for name in names:
        value = record.get(name.lower().replace("_", ""), "")
        if value not in ("None", ""):
            return value
    return default


# One query per securable level; each row is one principal's grant on that object.
PRIVILEGE_QUERIES = [
    ("CATALOG",
     f"SELECT * FROM {CATALOG}.information_schema.catalog_privileges", ()),
    ("SCHEMA",
     f"SELECT * FROM {CATALOG}.information_schema.schema_privileges", ("schema_name",)),
    ("TABLE",
     f"SELECT * FROM {CATALOG}.information_schema.table_privileges "
     f"WHERE table_schema = '{GOLD}'", ("table_name",)),
]

def grant_actions(privilege, object_type):
    """Actions a `privilege_type` confers on an object type.

    The two spellings have to collapse: `information_schema` reports Unity Catalog
    privileges underscored (`USE_CATALOG`, `EXTERNAL_USE_SCHEMA`, `READ_METADATA`),
    while `SHOW GRANTS` and GRANT syntax use spaces (`USE CATALOG`). `ALL PRIVILEGES`
    names no action at all and has to be expanded, or every principal holding it looks
    like it lacks each individual privilege.
    """
    action = " ".join(str(privilege).replace("_", " ").upper().split())
    if action in ("ALL PRIVILEGES", "ALL"):
        return {"USE", "SELECT", "MODIFY"}
    # USE CATALOG / USE SCHEMA / EXTERNAL USE SCHEMA collapse to USE + their object
    tokens = action.split()
    if len(tokens) > 1 and tokens[-1] == object_type:
        return {"USE"}
    return {action}


# principal -> {(action, object_type, object_key)}
grants_by_principal = {}
sources = 0
row_counts = {}
seen_types = {}

for object_type, query, key_columns in PRIVILEGE_QUERIES:
    try:
        rows = spark.sql(query).collect()
    except Exception as exc:
        note(f"{object_type.lower()} privilege view unavailable", str(exc).splitlines()[0][:160])
        continue
    sources += 1
    row_counts[object_type] = len(rows)
    for row in rows:
        record = normalised(row_dict(row))
        raw_privilege = field(record, "privilege_type")
        principal = field(record, "grantee", "principal")
        if not raw_privilege or not principal:
            continue
        seen_types.setdefault(object_type, set()).add(raw_privilege)
        if object_type == "CATALOG":
            key = CATALOG
        else:
            parts = [field(record, column) for column in key_columns]
            if not all(parts):
                continue
            key = f"{CATALOG}.{parts[0]}" if object_type == "SCHEMA" else parts[-1]
        for action in grant_actions(raw_privilege, object_type):
            grants_by_principal.setdefault(principal.lower(), set()).add(
                (action, object_type, key.lower()))

check("privilege views readable", sources == len(PRIVILEGE_QUERIES),
      f"{sources}/{len(PRIVILEGE_QUERIES)} loaded")
for object_type, _, _ in PRIVILEGE_QUERIES:
    types = ", ".join(sorted(seen_types.get(object_type, ()))) or "none"
    print(f"{object_type}: {row_counts.get(object_type, 0)} grant rows, privilege types [{types}]")
print(f"{len(grants_by_principal)} principals hold direct grants")

for role_name, spec in ROLES.items():
    grants = grants_by_principal.get(role_name.lower(), set())

    check(
        f"{role_name} has USE CATALOG",
        ("USE", "CATALOG", CATALOG) in grants,
        "" if ("USE", "CATALOG", CATALOG) in grants else "missing",
    )

    # Schema-level checks are subset-based: a missing expected grant is a gap, but
    # extra inherited grants (e.g. the catalog owner's ALL PRIVILEGES expanding to
    # rbac/default/information_schema) are a note, not a gap.
    actual_schemas = {key for action, object_type, key in grants
                      if action == "USE" and object_type == "SCHEMA"}
    expected_schemas = {f"{CATALOG}.{schema}" for schema in spec["schemas"]}
    check(
        f"{role_name} USE SCHEMA set",
        expected_schemas <= actual_schemas,
        f"missing {sorted(expected_schemas - actual_schemas)}",
    )
    extra_schemas = sorted(actual_schemas - expected_schemas)
    if extra_schemas:
        note(f"{role_name} holds USE SCHEMA beyond its catalog spec (inherited expansion)",
             ", ".join(extra_schemas))

    schema_selects = {key for action, object_type, key in grants
                      if action == "SELECT" and object_type == "SCHEMA"}
    schema_modifies = {key for action, object_type, key in grants
                       if action == "MODIFY" and object_type == "SCHEMA"}
    whole_layer = "ALL" in spec["domains"]
    expected_selects = {f"{CATALOG}.{schema}" for schema, privileges in spec["schemas"].items()
                        if "SELECT" in privileges} if whole_layer else set()
    expected_modifies = {f"{CATALOG}.{schema}" for schema, privileges in spec["schemas"].items()
                         if "MODIFY" in privileges} if whole_layer else set()

    check(f"{role_name} schema SELECT set", expected_selects <= schema_selects,
          f"missing {sorted(expected_selects - schema_selects)}")
    extra_selects = sorted(schema_selects - expected_selects)
    if extra_selects:
        note(f"{role_name} holds schema SELECT beyond its catalog spec", ", ".join(extra_selects))

    check(f"{role_name} schema MODIFY set", expected_modifies <= schema_modifies,
          f"missing {sorted(expected_modifies - schema_modifies)}")
    extra_modifies = sorted(schema_modifies - expected_modifies)
    if extra_modifies:
        note(f"{role_name} holds schema MODIFY beyond its catalog spec", ", ".join(extra_modifies))

    granted_tables = {key.split(".")[-1] for action, object_type, key in grants
                      if action == "SELECT" and object_type == "TABLE"}

    if "ALL" in spec["domains"]:
        if granted_tables:
            note(f"{role_name} has {len(granted_tables)} table-level grants on top of schema SELECT")
    else:
        allowed = set()
        for domain in spec["domains"]:
            allowed |= tables_by_domain.get(domain, set())
        check(f"{role_name} table SELECT set",
              granted_tables == allowed,
              f"missing {sorted(allowed - granted_tables)}, extra {sorted(granted_tables - allowed)}")

PASS  privilege views readable: 3/3 loaded
CATALOG: 20 grant rows, privilege types [ALL_PRIVILEGES, BROWSE, EXTERNAL_USE_SCHEMA, MANAGE, READ_METADATA, USE_CATALOG]
SCHEMA: 97 grant rows, privilege types [ALL_PRIVILEGES, CREATE_TABLE, EXTERNAL_USE_SCHEMA, MANAGE, MODIFY, READ_METADATA, SELECT, USE_SCHEMA]
TABLE: 456 grant rows, privilege types [ALL_PRIVILEGES, MANAGE, MODIFY, READ_METADATA, SELECT]
13 principals hold direct grants
PASS  cdr_admins has USE CATALOG
PASS  cdr_admins USE SCHEMA set: missing []
NOTE  cdr_admins holds USE SCHEMA beyond its catalog spec (inherited expansion): cdr_mimiciv.default, cdr_mimiciv.information_schema, cdr_mimiciv.rbac
PASS  cdr_admins schema SELECT set: missing []
NOTE  cdr_admins holds schema SELECT beyond its catalog spec: cdr_mimiciv.default, cdr_mimiciv.information_schema, cdr_mimiciv.rbac
PASS  cdr_admins schema MODIFY set: missing []
NOTE  cdr_admins holds schema MODIFY beyond its catalog spec: cdr_mimiciv.default, cdr_mimiciv.information_sche

## 5. Column masks

Tags classify, masks protect. A tagged column with no mask is a policy that
exists only on paper.

In [0]:
masked_columns = set()
try:
    for row in spark.sql(f"""
        SELECT * FROM {CATALOG}.information_schema.column_masks
        WHERE table_schema IN ({SCHEMA_LIST})
    """).collect():
        record = row_dict(row)
        masked_columns.add((
            pick(record, "table_schema").lower(),
            pick(record, "table_name").lower(),
            pick(record, "column_name").lower(),
        ))
    print(f"{len(masked_columns)} masked columns across {MASKED_SCHEMAS}")
except Exception as exc:
    note("column_masks unavailable", str(exc).splitlines()[0][:160])

# `deferred_columns` is a map keyed by schema; a flat list is the original gold-only form.
raw_deferred = MASKING.get("deferred_columns", {})
if isinstance(raw_deferred, dict):
    deferred = {(str(schema).lower(), str(column).lower())
                for schema, columns in raw_deferred.items() for column in columns}
else:
    deferred = {(GOLD.lower(), str(column).lower()) for column in raw_deferred}

exempt = {(
    str(entry.get("schema", GOLD)).lower(),
    str(entry["table"]).lower(),
    str(entry["column"]).lower(),
) for entry in MASKING.get("exempt_columns", [])}
protected = {}

for row in spark.sql(f"""
    SELECT c.table_schema, c.table_name, c.column_name, ct.tag_name, ct.tag_value
    FROM {CATALOG}.information_schema.columns c
    JOIN {CATALOG}.information_schema.tables t
      ON t.table_schema = c.table_schema AND t.table_name = c.table_name AND t.table_type <> 'VIEW'
    JOIN {CATALOG}.information_schema.column_tags ct
      ON ct.schema_name = c.table_schema AND ct.table_name = c.table_name
     AND ct.column_name = c.column_name
    WHERE c.table_schema IN ({SCHEMA_LIST})
""").collect():
    record = row_dict(row)
    tag_name, tag_value = record["tag_name"], record["tag_value"]
    hit = any(rule["tag"] == tag_name and rule["value"] == tag_value
              for rule in MASKING["columns"].values())
    if hit:
        protected[(
            record["table_schema"].lower(),
            record["table_name"].lower(),
            record["column_name"].lower(),
        )] = f"{tag_name}={tag_value}"

unmasked = sorted(
    key for key in protected
    if key not in masked_columns
    and key not in exempt
    and (key[0], key[2]) not in deferred)
deferred_hits = sorted(
    key for key in protected
    if key not in masked_columns
    and key not in exempt
    and (key[0], key[2]) in deferred)
exempt_hits = sorted(key for key in protected if key not in masked_columns and key in exempt)

for schema, table_name, column_name in exempt_hits:
    note(f"{schema}.{table_name}.{column_name} is tagged {protected[(schema, table_name, column_name)]} "
         f"but exempt from masking by policy (readable for every role)")
for schema, table_name, column_name in deferred_hits:
    note(f"{schema}.{table_name}.{column_name} protected by tag but masking is deferred "
         f"({protected[(schema, table_name, column_name)]})")

check("every tagged identifier column is masked", not unmasked,
      ", ".join(f"{s}.{t}.{c}" for s, t, c in unmasked[:10]) + (" ..." if len(unmasked) > 10 else ""))

78 masked columns across ['gold', 'silver']
NOTE  gold.dim_patient.anchor_age is tagged hipaa_identifiability=phi_quasi_identifier but exempt from masking by policy (readable for every role)
NOTE  gold.dim_patient.gender is tagged hipaa_identifiability=phi_quasi_identifier but exempt from masking by policy (readable for every role)
NOTE  gold.fact_admission.hadm_id protected by tag but masking is deferred (hipaa_identifiability=phi_quasi_identifier)
NOTE  gold.fact_diagnosis.hadm_id protected by tag but masking is deferred (hipaa_identifiability=phi_quasi_identifier)
NOTE  gold.fact_icu_observation.hadm_id protected by tag but masking is deferred (hipaa_identifiability=phi_quasi_identifier)
NOTE  gold.fact_icu_observation.stay_id protected by tag but masking is deferred (hipaa_identifiability=phi_quasi_identifier)
NOTE  gold.fact_icu_stay.hadm_id protected by tag but masking is deferred (hipaa_identifiability=phi_quasi_identifier)
NOTE  gold.fact_icu_stay.stay_id protected by tag but m

## 6. Row filters (RLS levels)

In [0]:
candidates = [c.lower() for c in CATALOG_DEF["rls"]["anchor_column_candidates"]]

gold_columns = {}
gold_table_types = {}
for row in spark.sql(f"""
    SELECT c.table_name, c.column_name, t.table_type
    FROM {CATALOG}.information_schema.columns c
    JOIN {CATALOG}.information_schema.tables t
      ON t.table_schema = c.table_schema AND t.table_name = c.table_name
    WHERE c.table_schema = '{GOLD}'
""").collect():
    record = row_dict(row)
    gold_columns.setdefault(record["table_name"], set()).add(record["column_name"].lower())
    gold_table_types[record["table_name"]] = record["table_type"].upper()

anchor_tables = {table for table, columns in gold_columns.items()
                 if gold_table_types.get(table) != "VIEW"
                 and any(candidate in columns for candidate in candidates)}

filtered_tables = set()
try:
    for row in spark.sql(f"SELECT * FROM {CATALOG}.information_schema.row_filters").collect():
        record = row_dict(row)
        if pick(record, "table_schema", "schema_name").lower() == GOLD:
            filtered_tables.add(pick(record, "table_name").lower())
except Exception as exc:
    note("row_filters unavailable", str(exc).splitlines()[0][:160])

check("every gold table with an RLS anchor column has a row filter",
      anchor_tables <= filtered_tables,
      f"missing {sorted(anchor_tables - filtered_tables)}")

# Id-anchored filters (`rls.id_anchored_filters`) carry the same cohort rule to the tables that have
# no service column. They are declared in the catalog, so a filter that has gone missing is a gap
# rather than a quiet widening of the cohort.
for key, spec in sorted(CATALOG_DEF["rls"].get("id_anchored_filters", {}).items()):
    anchor = str(spec["anchor_column"]).lower()
    built = {table.lower() for table, columns in gold_columns.items()
             if anchor in {column.lower() for column in columns}}
    declared = {str(table).lower() for table in spec["tables"]}
    not_built = sorted(declared - built)
    if not_built:
        note(f"id-anchored filter '{key}' declares tables not built with {anchor}", ", ".join(not_built))
    check(f"every {anchor} table carries the '{key}' row filter",
          (declared & built) <= filtered_tables,
          f"missing {sorted((declared & built) - filtered_tables)}")

uncovered = sorted(table for table in gold_columns
                   if table.lower() not in filtered_tables and table not in anchor_tables)
if uncovered:
    note(f"{len(uncovered)} gold tables have no cohort column, so levels 1/2 cannot filter them",
         ", ".join(uncovered[:10]) + (" ..." if len(uncovered) > 10 else ""))

PASS  every gold table with an RLS anchor column has a row filter: missing []
PASS  every hadm_id table carries the 'hadm' row filter: missing []
PASS  every subject_id table carries the 'subject' row filter: missing []
NOTE  15 gold tables have no cohort column, so levels 1/2 cannot filter them: dim_admission_type, dim_careunit, dim_date, dim_diagnosis, dim_icu_item, dim_lab_item, dim_location, dim_medication, dim_medication_route, dim_microbiology_test ...


In [0]:
print(f"\n{len(gaps)} gaps, {len(notes)} notes")
for gap in gaps:
    print("  -", gap)

if FAIL_ON_GAP and gaps:
    raise Exception(f"RBAC verification found {len(gaps)} gaps: {gaps}")

dbutils.notebook.exit(f"04_verify_rbac gaps={len(gaps)} notes={len(notes)}")